# 训练模型并保存可视化结果

这个 notebook 会完整跑一遍项目实验，并保存可被 `visualize_model_performance.ipynb` 读取的结果文件：

- `experiment_results.csv`: 核心策略评估结果
- `policy_sweep_results.csv`: 策略参数 sweep 结果
- `training_artifacts/training_metadata.json`: 训练配置、实际设备、模型信息
- `training_artifacts/reward_model_training_log.csv`: PyTorch reward model 的训练 loss，如果使用 PyTorch

默认会尝试使用 PyTorch reward model，并自动选择 `cuda` 或 `cpu`。如果当前环境没有 PyTorch，则退回到项目原本的 sklearn logistic reward model。

In [1]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_open_bandit_feedback, train_eval_split
from src.evaluation import evaluate_policy
from src.features import make_all_action_features, make_logged_action_features
from src.ope import fit_reward_model, predict_expected_rewards
from src.policies import (
    epsilon_greedy_popularity_policy,
    linear_thompson_sampling_policy,
    linucb_policy,
    logistic_ucb_policy,
    uniform_random_policy,
)

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    TORCH_AVAILABLE = False

from sklearn.preprocessing import StandardScaler

## 1. 配置

如果只是测试 notebook 流程，可以把 `MAX_ROUNDS` 设成一个较小数字，例如 `20000`。正式实验建议保留为 `None` 使用完整数据。

In [ ]:
SEED = 12345

BEHAVIOR_POLICY = "random"
CAMPAIGN = "all"
DATA_PATH = None
EVAL_SIZE = 0.5
MAX_ROUNDS = None  # example for a quick smoke run: 20000

USE_TORCH_REWARD_MODEL = True
USE_CUDA_IF_AVAILABLE = True
TORCH_EPOCHS = 800
TORCH_BATCH_SIZE = 4096
TORCH_PREDICT_BATCH_SIZE = 65536
TORCH_HIDDEN_DIM = 128
TORCH_LEARNING_RATE = 1e-3
TORCH_WEIGHT_DECAY = 1e-4

CORE_BOOTSTRAP_SAMPLES = 200
RUN_POLICY_SWEEP = True
SWEEP_BOOTSTRAP_SAMPLES = 0
INCLUDE_OBP_PARITY = True

EXPERIMENT_RESULTS_PATH = Path("experiment_results.csv")
POLICY_SWEEP_RESULTS_PATH = Path("policy_sweep_results.csv")
ARTIFACT_DIR = Path("training_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"PyTorch available: {TORCH_AVAILABLE}")
if TORCH_AVAILABLE:
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch available: True
PyTorch version: 2.5.1+cu124
CUDA available: True


In [3]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def subset_feedback(feedback: dict, max_rounds: int | None, seed: int) -> dict:
    if max_rounds is None or max_rounds >= feedback["n_rounds"]:
        return feedback
    rng = np.random.default_rng(seed)
    indices = rng.choice(feedback["n_rounds"], size=max_rounds, replace=False)
    return {
        "n_rounds": len(indices),
        "n_actions": feedback["n_actions"],
        "action": feedback["action"][indices],
        "position": feedback["position"][indices],
        "reward": feedback["reward"][indices],
        "pscore": feedback["pscore"][indices],
        "context": feedback["context"][indices],
        "action_context": feedback["action_context"],
    }


seed_everything(SEED)

## 2. 加载并切分 Open Bandit Dataset

In [4]:
feedback = load_open_bandit_feedback(
    behavior_policy=BEHAVIOR_POLICY,
    campaign=CAMPAIGN,
    data_path=DATA_PATH,
)
feedback = subset_feedback(feedback, MAX_ROUNDS, SEED)
train_feedback, eval_feedback = train_eval_split(feedback, eval_size=EVAL_SIZE, seed=SEED)

dataset_summary = {
    "n_rounds": feedback["n_rounds"],
    "train_rounds": train_feedback["n_rounds"],
    "eval_rounds": eval_feedback["n_rounds"],
    "n_actions": feedback["n_actions"],
    "context_dim": int(feedback["context"].shape[1]),
    "action_context_dim": int(feedback["action_context"].shape[1]),
    "behavior_mean_reward_eval": float(eval_feedback["reward"].mean()),
}
dataset_summary

INFO:obp.dataset.real:When `data_path` is not given, this class downloads the small-sized version of Open Bandit Dataset.


{'n_rounds': 10000,
 'train_rounds': 5000,
 'eval_rounds': 5000,
 'n_actions': 80,
 'context_dim': 20,
 'action_context_dim': 4,
 'behavior_mean_reward_eval': 0.004}

## 3. 训练 reward model

Reward model 负责估计 `q_hat(x, a) = E[r | x, a]`，随后 DM 和 DR 会使用这个预测矩阵。

In [5]:
if TORCH_AVAILABLE:
    class TorchRewardNet(nn.Module):
        def __init__(self, input_dim: int, hidden_dim: int) -> None:
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.15),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1),
            )

        def forward(self, x):
            return self.net(x).squeeze(-1)


def fit_predict_torch_reward_model(train_feedback: dict, eval_feedback: dict, device: str):
    x_train = make_logged_action_features(train_feedback).astype(np.float32)
    y_train = train_feedback["reward"].astype(np.float32)

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train).astype(np.float32)

    input_dim = x_train.shape[1]
    model = TorchRewardNet(input_dim=input_dim, hidden_dim=TORCH_HIDDEN_DIM).to(device)

    positives = float(y_train.sum())
    negatives = float(y_train.shape[0] - positives)
    pos_weight_value = negatives / max(positives, 1.0)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=TORCH_LEARNING_RATE, weight_decay=TORCH_WEIGHT_DECAY)

    dataset = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))
    loader = DataLoader(dataset, batch_size=TORCH_BATCH_SIZE, shuffle=True, drop_last=False)

    history = []
    start_time = time.time()
    for epoch in range(1, TORCH_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_examples = 0
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item()) * xb.shape[0]
            total_examples += xb.shape[0]

        epoch_loss = total_loss / max(total_examples, 1)
        history.append({"epoch": epoch, "loss": epoch_loss, "device": device})
        print(f"epoch {epoch:02d}/{TORCH_EPOCHS} - loss={epoch_loss:.6f}")

    all_features = make_all_action_features(eval_feedback["context"], eval_feedback["action_context"])
    n_rounds, n_actions, n_features = all_features.shape
    flat_features = all_features.reshape(n_rounds * n_actions, n_features)
    q_hat_flat = np.empty(flat_features.shape[0], dtype=np.float32)

    model.eval()
    with torch.no_grad():
        for start in range(0, flat_features.shape[0], TORCH_PREDICT_BATCH_SIZE):
            end = min(start + TORCH_PREDICT_BATCH_SIZE, flat_features.shape[0])
            batch = scaler.transform(flat_features[start:end]).astype(np.float32)
            xb = torch.from_numpy(batch).to(device, non_blocking=True)
            q_hat_flat[start:end] = torch.sigmoid(model(xb)).cpu().numpy()

    elapsed_seconds = time.time() - start_time
    q_hat = q_hat_flat.reshape(n_rounds, n_actions)
    return model, scaler, q_hat, pd.DataFrame(history), elapsed_seconds, pos_weight_value

In [6]:
device = "cpu"
if TORCH_AVAILABLE and USE_TORCH_REWARD_MODEL:
    if USE_CUDA_IF_AVAILABLE and torch.cuda.is_available():
        device = "cuda"
    reward_model_backend = "torch"
    reward_model, reward_scaler, q_hat, training_log, reward_model_seconds, pos_weight = fit_predict_torch_reward_model(
        train_feedback,
        eval_feedback,
        device=device,
    )
    training_log.to_csv(ARTIFACT_DIR / "reward_model_training_log.csv", index=False)
    torch.save(
        {
            "model_state_dict": reward_model.state_dict(),
            "input_dim": make_logged_action_features(train_feedback).shape[1],
            "hidden_dim": TORCH_HIDDEN_DIM,
            "scaler_mean": reward_scaler.mean_,
            "scaler_scale": reward_scaler.scale_,
            "device_used_for_training": device,
        },
        ARTIFACT_DIR / "torch_reward_model.pt",
    )
else:
    reward_model_backend = "sklearn_logistic_regression"
    reward_model_start = time.time()
    reward_model = fit_reward_model(train_feedback)
    q_hat = predict_expected_rewards(reward_model, eval_feedback)
    reward_model_seconds = time.time() - reward_model_start
    training_log = pd.DataFrame()
    pos_weight = None

print(f"reward_model_backend={reward_model_backend}")
print(f"device={device}")
print(f"q_hat shape={q_hat.shape}, min={q_hat.min():.6f}, max={q_hat.max():.6f}, mean={q_hat.mean():.6f}")

epoch 01/8 - loss=1.388476
epoch 02/8 - loss=1.386549
epoch 03/8 - loss=1.376153
epoch 04/8 - loss=1.359185
epoch 05/8 - loss=1.354210
epoch 06/8 - loss=1.346634
epoch 07/8 - loss=1.332106
epoch 08/8 - loss=1.322855
reward_model_backend=torch
device=cuda
q_hat shape=(5000, 80), min=0.137251, max=0.559129, mean=0.464948


## 4. 评估核心策略并保存 `experiment_results.csv`

In [7]:
def build_core_policies(train_feedback: dict, eval_feedback: dict):
    policies = [("uniform_random", uniform_random_policy(eval_feedback))]

    eps_policy, best_action = epsilon_greedy_popularity_policy(
        train_feedback,
        eval_feedback,
        epsilon=0.2,
    )
    policies.append((f"epsilon_greedy_popularity_best_action_{best_action}", eps_policy))
    policies.append(("linucb_alpha_0.5", linucb_policy(train_feedback, eval_feedback, alpha=0.5)))
    policies.append(("logistic_ucb_alpha_0.5", logistic_ucb_policy(train_feedback, eval_feedback, alpha=0.5)))
    policies.append(
        (
            "linear_thompson_sampling",
            linear_thompson_sampling_policy(train_feedback, eval_feedback, sample_scale=0.2, seed=SEED),
        )
    )
    return policies


core_rows = []
for policy_name, action_dist in build_core_policies(train_feedback, eval_feedback):
    print(f"evaluating {policy_name}")
    row = evaluate_policy(
        name=policy_name,
        action_dist=action_dist,
        eval_feedback=eval_feedback,
        q_hat=q_hat,
        bootstrap_samples=CORE_BOOTSTRAP_SAMPLES,
        alpha=0.05,
        seed=SEED,
        include_obp=INCLUDE_OBP_PARITY,
    )
    row["reward_model_backend"] = reward_model_backend
    row["device"] = device
    core_rows.append(row)

core_results = pd.DataFrame(core_rows)
core_results.to_csv(EXPERIMENT_RESULTS_PATH, index=False)
display(core_results)
print(f"saved {EXPERIMENT_RESULTS_PATH}")

evaluating uniform_random
evaluating epsilon_greedy_popularity_best_action_49
evaluating linucb_alpha_0.5
evaluating linear_thompson_sampling


,policy,behavior_mean_reward,ips,snips,dm,dr,logged_action_match_rate,mean_importance_weight,effective_sample_size,effective_sample_size_ratio,...,obp_ipw,obp_ipw_abs_diff,obp_snipw,obp_snipw_abs_diff,obp_dm,obp_dm_abs_diff,obp_dr,obp_dr_abs_diff,reward_model_backend,device
0,uniform_random,0.004,0.0040,0.004000,0.464948,0.003922,1.0000,1.000,5000.00000,1.000000,...,0.0040,0.0,0.004000,0.0,0.464948,5.551115e-17,0.003922,5.724587e-17,torch,cuda
1,epsilon_greedy_popularity_best_action_49,0.004,0.0136,0.015044,0.454258,0.052440,1.0000,0.904,90.04619,0.018009,...,0.0136,0.0,0.015044,0.0,0.454258,1.665335e-16,0.052440,5.551115e-17,torch,cuda
2,linucb_alpha_0.5,0.004,0.0160,0.017857,0.469532,0.061049,0.0112,0.896,56.00000,0.011200,...,0.0160,0.0,0.017857,0.0,0.469532,0.000000e+00,0.061049,0.000000e+00,torch,cuda
3,linear_thompson_sampling,0.004,0.0000,0.000000,0.470154,0.016178,0.0120,0.960,60.00000,0.012000,...,0.0000,0.0,0.000000,0.0,0.470154,0.000000e+00,0.016178,0.000000e+00,torch,cuda


saved experiment_results.csv


## 5. 参数 sweep 并保存 `policy_sweep_results.csv`

默认 sweep 不做 bootstrap，以免运行时间太长。需要置信区间时可以把 `SWEEP_BOOTSTRAP_SAMPLES` 改成 100 或更高。

In [8]:
def build_policy_grid(train_feedback: dict, eval_feedback: dict, seed: int):
    yield ("uniform_random", "uniform_random", "none", 0.0, uniform_random_policy(eval_feedback))

    for epsilon in [0.0, 0.05, 0.1, 0.2, 0.4, 0.8, 1.0]:
        action_dist, best_action = epsilon_greedy_popularity_policy(train_feedback, eval_feedback, epsilon=epsilon)
        yield (
            f"epsilon_greedy_popularity_epsilon_{epsilon:g}_best_action_{best_action}",
            "epsilon_greedy_popularity",
            "epsilon",
            epsilon,
            action_dist,
        )

    for alpha in [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]:
        yield (f"linucb_alpha_{alpha:g}", "linucb", "alpha", alpha, linucb_policy(train_feedback, eval_feedback, alpha=alpha))

    for alpha in [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]:
        yield (
            f"logistic_ucb_alpha_{alpha:g}",
            "logistic_ucb",
            "alpha",
            alpha,
            logistic_ucb_policy(train_feedback, eval_feedback, alpha=alpha),
        )

    for sample_scale in [0.05, 0.1, 0.2, 0.5, 1.0]:
        yield (
            f"linear_thompson_sampling_scale_{sample_scale:g}",
            "linear_thompson_sampling",
            "sample_scale",
            sample_scale,
            linear_thompson_sampling_policy(train_feedback, eval_feedback, sample_scale=sample_scale, seed=seed),
        )


if RUN_POLICY_SWEEP:
    sweep_rows = []
    for policy_name, family, parameter, value, action_dist in build_policy_grid(train_feedback, eval_feedback, SEED):
        print(f"sweep evaluating {policy_name}")
        row = evaluate_policy(
            name=policy_name,
            action_dist=action_dist,
            eval_feedback=eval_feedback,
            q_hat=q_hat,
            bootstrap_samples=SWEEP_BOOTSTRAP_SAMPLES,
            alpha=0.05,
            seed=SEED,
            include_obp=INCLUDE_OBP_PARITY,
        )
        row["policy_family"] = family
        row["sweep_parameter"] = parameter
        row["sweep_value"] = value
        row["reward_model_backend"] = reward_model_backend
        row["device"] = device
        sweep_rows.append(row)

    sweep_results = pd.DataFrame(sweep_rows)
    first_columns = ["policy_family", "sweep_parameter", "sweep_value", "policy"]
    sweep_results = sweep_results[first_columns + [col for col in sweep_results.columns if col not in first_columns]]
    sweep_results.to_csv(POLICY_SWEEP_RESULTS_PATH, index=False)
    display(sweep_results.head())
    print(f"saved {POLICY_SWEEP_RESULTS_PATH}")
else:
    sweep_results = pd.DataFrame()
    print("policy sweep skipped")

sweep evaluating uniform_random
sweep evaluating epsilon_greedy_popularity_epsilon_0_best_action_49
sweep evaluating epsilon_greedy_popularity_epsilon_0.05_best_action_49
sweep evaluating epsilon_greedy_popularity_epsilon_0.1_best_action_49
sweep evaluating epsilon_greedy_popularity_epsilon_0.2_best_action_49
sweep evaluating epsilon_greedy_popularity_epsilon_0.4_best_action_49
sweep evaluating epsilon_greedy_popularity_epsilon_0.8_best_action_49
sweep evaluating epsilon_greedy_popularity_epsilon_1_best_action_49
sweep evaluating linucb_alpha_0
sweep evaluating linucb_alpha_0.1
sweep evaluating linucb_alpha_0.25
sweep evaluating linucb_alpha_0.5
sweep evaluating linucb_alpha_1
sweep evaluating linucb_alpha_2
sweep evaluating linear_thompson_sampling_scale_0.05
sweep evaluating linear_thompson_sampling_scale_0.1
sweep evaluating linear_thompson_sampling_scale_0.2
sweep evaluating linear_thompson_sampling_scale_0.5
sweep evaluating linear_thompson_sampling_scale_1


,policy_family,sweep_parameter,sweep_value,policy,behavior_mean_reward,ips,snips,dm,dr,logged_action_match_rate,...,obp_ipw,obp_ipw_abs_diff,obp_snipw,obp_snipw_abs_diff,obp_dm,obp_dm_abs_diff,obp_dr,obp_dr_abs_diff,reward_model_backend,device
0,uniform_random,none,0.00,uniform_random,0.004,0.0040,0.004000,0.464948,0.003922,1.000,...,0.0040,0.0,0.004000,0.000000e+00,0.464948,5.551115e-17,0.003922,5.724587e-17,torch,cuda
1,epsilon_greedy_popularity,epsilon,0.00,epsilon_greedy_popularity_epsilon_0_best_actio...,0.004,0.0160,0.018182,0.451585,0.064570,0.011,...,0.0160,0.0,0.018182,0.000000e+00,0.451585,0.000000e+00,0.064570,0.000000e+00,torch,cuda
2,epsilon_greedy_popularity,epsilon,0.05,epsilon_greedy_popularity_epsilon_0.05_best_ac...,0.004,0.0154,0.017381,0.452253,0.061537,1.000,...,0.0154,0.0,0.017381,3.469447e-18,0.452253,1.110223e-16,0.061537,4.163336e-17,torch,cuda
3,epsilon_greedy_popularity,epsilon,0.10,epsilon_greedy_popularity_epsilon_0.1_best_act...,0.004,0.0148,0.016592,0.452921,0.058505,1.000,...,0.0148,0.0,0.016592,0.000000e+00,0.452921,5.551115e-17,0.058505,5.551115e-17,torch,cuda
4,epsilon_greedy_popularity,epsilon,0.20,epsilon_greedy_popularity_epsilon_0.2_best_act...,0.004,0.0136,0.015044,0.454258,0.052440,1.000,...,0.0136,0.0,0.015044,0.000000e+00,0.454258,1.665335e-16,0.052440,5.551115e-17,torch,cuda


saved policy_sweep_results.csv


## 6. 保存训练 metadata

In [9]:
metadata = {
    "seed": SEED,
    "behavior_policy": BEHAVIOR_POLICY,
    "campaign": CAMPAIGN,
    "eval_size": EVAL_SIZE,
    "max_rounds": MAX_ROUNDS,
    "dataset_summary": dataset_summary,
    "reward_model_backend": reward_model_backend,
    "torch_available": TORCH_AVAILABLE,
    "cuda_available": bool(torch.cuda.is_available()) if TORCH_AVAILABLE else False,
    "device_used": device,
    "torch_version": torch.__version__ if TORCH_AVAILABLE else None,
    "torch_epochs": TORCH_EPOCHS if reward_model_backend == "torch" else None,
    "torch_batch_size": TORCH_BATCH_SIZE if reward_model_backend == "torch" else None,
    "torch_hidden_dim": TORCH_HIDDEN_DIM if reward_model_backend == "torch" else None,
    "torch_learning_rate": TORCH_LEARNING_RATE if reward_model_backend == "torch" else None,
    "torch_pos_weight": pos_weight,
    "reward_model_seconds": reward_model_seconds,
    "core_bootstrap_samples": CORE_BOOTSTRAP_SAMPLES,
    "run_policy_sweep": RUN_POLICY_SWEEP,
    "sweep_bootstrap_samples": SWEEP_BOOTSTRAP_SAMPLES,
    "include_obp_parity": INCLUDE_OBP_PARITY,
    "outputs": {
        "experiment_results": str(EXPERIMENT_RESULTS_PATH),
        "policy_sweep_results": str(POLICY_SWEEP_RESULTS_PATH) if RUN_POLICY_SWEEP else None,
        "artifact_dir": str(ARTIFACT_DIR),
    },
}

metadata_path = ARTIFACT_DIR / "training_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"saved {metadata_path}")
metadata

saved training_artifacts\training_metadata.json


{'seed': 12345,
 'behavior_policy': 'random',
 'campaign': 'all',
 'eval_size': 0.5,
 'max_rounds': None,
 'dataset_summary': {'n_rounds': 10000,
  'train_rounds': 5000,
  'eval_rounds': 5000,
  'n_actions': 80,
  'context_dim': 20,
  'action_context_dim': 4,
  'behavior_mean_reward_eval': 0.004},
 'reward_model_backend': 'torch',
 'torch_available': True,
 'cuda_available': True,
 'device_used': 'cuda',
 'torch_version': '2.5.1+cu124',
 'torch_epochs': 8,
 'torch_batch_size': 4096,
 'torch_hidden_dim': 128,
 'torch_learning_rate': 0.001,
 'torch_pos_weight': 276.77777777777777,
 'reward_model_seconds': 1.5043625831604004,
 'core_bootstrap_samples': 200,
 'run_policy_sweep': True,
 'sweep_bootstrap_samples': 0,
 'include_obp_parity': True,
 'outputs': {'experiment_results': 'experiment_results.csv',
  'policy_sweep_results': 'policy_sweep_results.csv',
  'artifact_dir': 'training_artifacts'}}

## 7. 快速检查结果

下面只做一个轻量 sanity check。完整图表请打开 `visualize_model_performance.ipynb` 并重跑所有 cells。

In [10]:
ranking_metric = "dr" if "dr" in core_results.columns else "ips"
summary = core_results[["policy", ranking_metric, "behavior_mean_reward", "logged_action_match_rate", "mean_importance_weight"]].copy()
summary["relative_to_behavior"] = summary[ranking_metric] / summary["behavior_mean_reward"]
summary = summary.sort_values(ranking_metric, ascending=False)
display(summary)

print("Done. Open visualize_model_performance.ipynb to generate the report figures.")

,policy,dr,behavior_mean_reward,logged_action_match_rate,mean_importance_weight,relative_to_behavior
2,linucb_alpha_0.5,0.061049,0.004,0.0112,0.896,15.262282
1,epsilon_greedy_popularity_best_action_49,0.052440,0.004,1.0000,0.904,13.110054
3,linear_thompson_sampling,0.016178,0.004,0.0120,0.960,4.044456
0,uniform_random,0.003922,0.004,1.0000,1.000,0.980596


Done. Open visualize_model_performance.ipynb to generate the report figures.
